In [1]:
from pyspark.sql.functions import \
    col, current_date, dayofweek, datediff, floor, \
    hour, log, month, months_between, pow, sqrt
from pyspark.ml.feature import StringIndexer


# Needed columns to train the ML model
FEATURE_COLS = [
    "amt",
    "log_amt",
    "city_pop",
    "customer_age",
    "distance_customer_merchant",
    "trans_hour",
    "trans_dayofweek",
    "trans_month",
    "category_index"
]

# Row indentification columns
ID_COLS = [
    "trans_num",
    "trans_date_trans_time",
    "cc_num",
]

# Target column for prediction results
LABEL_COL = "is_fraud"


def build_features(cleaned_table):
    df = spark.sql(f"SELECT * FROM fraud_detection_lakehouse.dbo.{cleaned_table}")

    # Split transaction times
    df = (
        df
        .withColumn("trans_hour", hour("trans_date_trans_time"))
        .withColumn("trans_dayofweek", dayofweek("trans_date_trans_time"))
        .withColumn("trans_month", month("trans_date_trans_time"))
    )

    # Extract customer age
    df = df.withColumn(
        "customer_age",
        floor(months_between(current_date(), col("dob")) / 12)
    )
    
    # Calculate how far the merchant is from the client
    df = df.withColumn(
        "distance_customer_merchant",
        sqrt(
            pow(df.lat - df.merch_lat, 2) +
            pow(df.long - df.merch_long, 2)
        )
    )

    # Create logarithm transaction amount so that 
    # its easier for the model to evaluate the differences
    df = df.withColumn("log_amt", log(df.amt + 1))

    return df

StatementMeta(, c8fab459-d90f-4472-bddc-d34feaf1bf7b, 3, Finished, Available, Finished)

### Build test and train tables features

In [2]:
df_test_feat = build_features("cleaned_test_transactions")
df_train_feat = build_features("cleaned_train_transactions")

StatementMeta(, c8fab459-d90f-4472-bddc-d34feaf1bf7b, 4, Finished, Available, Finished)

### Apply Features

In [3]:
# Convert the merchant categories to numbers
category_indexer = StringIndexer(
    inputCol="category",
    outputCol="category_index",
    handleInvalid="keep"
)

# Create category dictionary from train data
category_model = category_indexer.fit(df_train_feat)

# Apply the category dictionary changes
df_train_feat = category_model.transform(df_train_feat)
df_test_feat = category_model.transform(df_test_feat)

# Create ML ready tables
df_train_gold = df_train_feat.select(ID_COLS + FEATURE_COLS + [LABEL_COL])
df_test_gold = df_test_feat.select(ID_COLS + FEATURE_COLS + [LABEL_COL])

StatementMeta(, c8fab459-d90f-4472-bddc-d34feaf1bf7b, 5, Finished, Available, Finished)

### Write ML tables

In [4]:
df_train_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ml_train_features")
df_test_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ml_test_features")

StatementMeta(, c8fab459-d90f-4472-bddc-d34feaf1bf7b, 6, Finished, Available, Finished)